# تمرین ۳ — مترجم فارسی به انگلیسی با Transformer

**انجام‌دهنده: مسعود کارگر**  |  درس: یادگیری عمیق / پردازش زبان طبیعی

> حالت پیش‌فرض از داده‌های واقعی استفاده می‌کند. برای بررسی سریع بدون اینترنت فقط متغیر محیطی `NB_OFFLINE_SMOKE=1` را قبل از شروع نوت‌بوک تنظیم کنید. در این حالت، خروجی‌ها صرفاً آزمون عملکرد کد هستند و **معیار عملکرد روی دیتاست اصلی محسوب نمی‌شوند**.

منبع: [Shenasa English-Persian Parallel Dataset](https://huggingface.co/datasets/shenasa/English-Persian-Parallel-Dataset)؛ در حالت واقعی زیرمجموعه‌ای با حداکثر **۱۰۰٬۰۰۰ زوج‌جمله** خوانده می‌شود. تقسیم داده به آموزش، اعتبارسنجی و آزمون پیش از ساخت واژگان انجام می‌شود.

In [1]:
import os, random, re, time, io, tarfile, urllib.request
from collections import Counter
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(2,torch.get_num_threads()))
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SMOKE=os.getenv('NB_OFFLINE_SMOKE','0')=='1'
print('PyTorch:',torch.__version__,'| device:',DEVICE,'| mode:', 'OFFLINE SMOKE TEST' if SMOKE else 'REAL DATASET')

PyTorch: 2.15.0+cu132 | device: cpu | mode: REAL DATASET


برای اجرا در Colab در صورت نبود کتابخانهٔ datasets ابتدا دستور زیر را اجرا کنید: ` %pip install -q datasets `

In [2]:
import importlib.util
if not SMOKE and importlib.util.find_spec('datasets') is None:
    %pip install -q datasets


In [3]:
# Source dataset: shenasa/English-Persian-Parallel-Dataset, streaming subset.
# This target size is comparable to the common ~100k sentence-pair small translation tutorials.
DATASET_ID='shenasa/English-Persian-Parallel-Dataset'
N_PAIRS=100000 if not SMOKE else 64
if SMOKE:
    samples=[('سلام','hello'),('حال شما چطور است ؟','how are you ?'),('من دانشجو هستم .','i am a student .'),('این کتاب خوب است .','this book is good .'),('هوا امروز خوب است .','the weather is nice today .'),('صبح بخیر','good morning'),('متشکرم','thank you'),('نام من مسعود است .','my name is masoud .')]
    pairs=(samples*8)[:N_PAIRS]
else:
    try:from datasets import load_dataset
    except ImportError:raise ImportError('Install datasets first: %pip install datasets')
    stream=load_dataset(DATASET_ID,split='train',streaming=True)
    first=next(iter(stream));print('Original columns:',list(first.keys()))
    # The dataset's exported TSV may use its first bilingual example as headers.
    column_names=list(first.keys())
    normalized={k.lower().strip():k for k in column_names}
    en_key=next((normalized[x] for x in ('english','en','english_sentence','source') if x in normalized),None)
    fa_key=next((normalized[x] for x in ('persian','farsi','fa','persian_sentence','target') if x in normalized),None)
    if en_key is None or fa_key is None:
        if len(column_names)!=2:
            raise ValueError(f'Cannot infer English/Persian columns: {column_names}')
        en_key,fa_key=column_names[0],column_names[1]
    print('Mapped fields: English =',repr(en_key),'| Persian =',repr(fa_key))
    pairs=[]; seen=set()
    for row in stream:
        fa,en=row.get(fa_key),row.get(en_key)
        if not isinstance(fa,str) or not isinstance(en,str):continue
        fa,en=fa.strip(),en.strip()
        if 2<=len(fa)<=120 and 2<=len(en)<=150 and (fa,en) not in seen:
            seen.add((fa,en));pairs.append((fa,en))
            if len(pairs)>=N_PAIRS:break
    if len(pairs)<10000:raise ValueError(f'Not enough usable pairs: {len(pairs)}, columns={column_names}')
random.Random(SEED).shuffle(pairs)
test_n=max(4,int(.05*len(pairs)));val_n=max(4,int(.05*len(pairs)))
test_pairs=pairs[:test_n];val_pairs=pairs[test_n:test_n+val_n];train_pairs=pairs[test_n+val_n:]
print('Train/val/test:',len(train_pairs),len(val_pairs),len(test_pairs));print('Example:',train_pairs[0])
# Character-level tokenization avoids external pretrained tokenizers and handles Persian directly.
PAD,BOS,EOS,UNK=0,1,2,3
fa_chars=sorted(set(''.join(x for x,y in train_pairs)))
en_chars=sorted(set(''.join(y for x,y in train_pairs)))
fa_vocab={c:i+4 for i,c in enumerate(fa_chars)}
en_vocab={c:i+4 for i,c in enumerate(en_chars)}
en_inverse={i:c for c,i in en_vocab.items()}
SRC_LEN=60 if SMOKE else 120; TGT_LEN=64 if SMOKE else 130

def fa_encode(x):return [fa_vocab.get(c,UNK) for c in x[:SRC_LEN-1]]+[EOS]
def en_encode(y):return [en_vocab.get(c,UNK) for c in y[:TGT_LEN-2]]+[EOS]
class TranslationDataset(Dataset):
    def __init__(self,items):self.items=items
    def __len__(self):return len(self.items)
    def __getitem__(self,i):
        fa,en=self.items[i];return torch.tensor(fa_encode(fa)),torch.tensor([BOS]+en_encode(en))
def pad_batch(items):
    a,b=zip(*items)
    return nn.utils.rnn.pad_sequence(a,batch_first=True,padding_value=PAD),nn.utils.rnn.pad_sequence(b,batch_first=True,padding_value=PAD)
BATCH=16 if SMOKE else 64
train_dl=DataLoader(TranslationDataset(train_pairs),batch_size=BATCH,shuffle=True,collate_fn=pad_batch)
val_dl=DataLoader(TranslationDataset(val_pairs),batch_size=BATCH,collate_fn=pad_batch)
print('vocab Persian/English:',len(fa_vocab)+4,len(en_vocab)+4)

/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Original columns: ['flash fire .', 'فلاش آتش .']
Mapped fields: English = 'flash fire .' | Persian = 'فلاش آتش .'


Train/val/test: 90000 5000 5000
Example: ('آیا این کافی نیست که من بلعیده شوم ، بدون اینکه انتظار داشته باشم قدرتی که مرا می بلعد برکت دهد ؟', 'Is it not enough that I am devoured, without my being expected to bless the power that devours me?')
vocab Persian/English: 127 85


In [4]:
class Seq2SeqTransformer(nn.Module):
    def __init__(self,n_source,n_target,d=64,heads=4,layers=2):
        super().__init__()
        self.source_emb=nn.Embedding(n_source,d,padding_idx=PAD)
        self.target_emb=nn.Embedding(n_target,d,padding_idx=PAD)
        self.src_pos=nn.Embedding(SRC_LEN,d);self.tgt_pos=nn.Embedding(TGT_LEN,d)
        self.transformer=nn.Transformer(d_model=d,nhead=heads,num_encoder_layers=layers,num_decoder_layers=layers,dim_feedforward=d*4,dropout=.1,batch_first=True,norm_first=True)
        self.head=nn.Linear(d,n_target)
    def forward(self,src,tgt):
        bs,ss=src.shape;bt,tt=tgt.shape
        se=self.source_emb(src)+self.src_pos(torch.arange(ss,device=src.device))[None]
        te=self.target_emb(tgt)+self.tgt_pos(torch.arange(tt,device=tgt.device))[None]
        causal=torch.triu(torch.ones((tt,tt),device=tgt.device,dtype=torch.bool),diagonal=1)
        result=self.transformer(se,te,tgt_mask=causal,src_key_padding_mask=src.eq(PAD),tgt_key_padding_mask=tgt.eq(PAD),memory_key_padding_mask=src.eq(PAD))
        return self.head(result)
model=Seq2SeqTransformer(len(fa_vocab)+4,len(en_vocab)+4).to(DEVICE)
optimizer=torch.optim.AdamW(model.parameters(),lr=.001)
print('parameters:',sum(p.numel() for p in model.parameters()))

/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/torch/nn/modules/transformer.py:145: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


parameters: 268821


In [5]:
loss_fn=nn.CrossEntropyLoss(ignore_index=PAD)
EPOCHS=2 if SMOKE else 8
def run_epoch(loader,training):
    model.train(training);total=count=0
    for src,tgt in loader:
        src,tgt=src.to(DEVICE),tgt.to(DEVICE)
        with torch.set_grad_enabled(training):
            logits=model(src,tgt[:,:-1]);expected=tgt[:,1:]
            loss=loss_fn(logits.reshape(-1,logits.size(-1)),expected.reshape(-1))
            if training:
                optimizer.zero_grad();loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.0);optimizer.step()
        total+=loss.item();count+=1
    return total/max(1,count)
for e in range(EPOCHS):
    tr=run_epoch(train_dl,True);vl=run_epoch(val_dl,False)
    print(f'epoch {e+1}: train_loss={tr:.3f}, val_loss={vl:.3f}')

epoch 1: train_loss=2.266, val_loss=1.930


epoch 2: train_loss=1.911, val_loss=1.744


epoch 3: train_loss=1.793, val_loss=1.655


epoch 4: train_loss=1.726, val_loss=1.598


epoch 5: train_loss=1.682, val_loss=1.556


epoch 6: train_loss=1.650, val_loss=1.531


epoch 7: train_loss=1.625, val_loss=1.502


epoch 8: train_loss=1.604, val_loss=1.479


In [6]:
@torch.no_grad()
def translate(fa,max_new=90):
    model.eval()
    src=torch.tensor([fa_encode(fa)],device=DEVICE)
    generated=[BOS]
    for _ in range(min(max_new,TGT_LEN-1)):
        tgt=torch.tensor([generated],device=DEVICE)
        scores=model(src,tgt)[0,-1]
        scores[PAD]=-1e9;scores[BOS]=-1e9
        nxt=int(scores.argmax().item())
        if nxt==EOS:break
        generated.append(nxt)
    return ''.join(en_inverse.get(i,'?') for i in generated[1:])
tests=['سلام','حال شما چطور است ؟','من دانشجو هستم .','هوا امروز خوب است .','نام من مسعود است .']
for s in tests:print(f'FA: {s}\nEN (model): {translate(s)}\n')
if SMOKE:print('SMOKE TEST: the model was trained on 64 toy pairs. Do not report translations as real-dataset quality.')

FA: سلام
EN (model): His

FA: حال شما چطور است ؟
EN (model): Now you are you are you?

FA: من دانشجو هستم .
EN (model): I am all the stand.

FA: هوا امروز خوب است .
EN (model): He was a started to the start.

FA: نام من مسعود است .
EN (model): I don't me the start.



## محدودیت‌ها

این مدل نویسه‌محور از صفر آموزش می‌بیند؛ انتظار کیفیت ترجمهٔ مدل‌های پیش‌آموزش‌دیدهٔ بزرگ را نباید داشت. برای ارزیابی رسمی می‌توان BLEU/chrF را روی مجموعه آزمون ثابت محاسبه کرد. داده‌های مجموعه ممکن است محاوره‌ای، کوتاه یا دارای نویز باشند.